# Python reference 6: iterators, generators and itertools

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-06-iterators-and-itertools.ipynb)

Iterables versus one-shot iterators, built-in helpers (`zip(strict=True)`, `enumerate`, `any`/`all`), comprehensions, generators and lazy pipelines, and the `itertools` toolkit (`islice`, `chain`, `groupby`, `pairwise`, `batched`, `accumulate`, combinatorics). Practise it in [Python practice 6: iterators, generators and itertools](../practice/python-practice-06-iterators-and-itertools.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [The iteration protocol](#the-iteration-protocol) | [Iterables and iterators](#iterables-and-iterators), [Built-in iteration helpers](#built-in-iteration-helpers) |
| [Comprehensions](#comprehensions) | [List, dict, set and generator comprehensions](#list-dict-set-and-generator-comprehensions) |
| [Generators](#generators) | [Generator functions and `yield`](#generator-functions-and-yield), [Lazy pipelines](#lazy-pipelines) |
| [itertools](#itertools) | [Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`](#slicing-and-chaining-islice-chain-takewhile-dropwhile), [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](#grouping-and-windows-groupby-pairwise-batched-accumulate), [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](#combinatorics-product-permutations-combinations-zip_longest-starmap-tee) |

## The iteration protocol

### Iterables and iterators

An **iterable** is anything `for` can loop over: it has `__iter__` (or `__getitem__`). `iter(iterable)` returns an
**iterator**, which has `__next__` and raises `StopIteration` when exhausted. Lists, dicts, strings and files are
iterables; files, generators, `map`, `zip` and `enumerate` objects are iterators.

| call | returns |
| --- | --- |
| `iter(obj)` | an iterator over `obj` |
| `next(it[, default])` | the next item; `default` instead of `StopIteration` when exhausted |
| `iter(callable, sentinel)` | calls `callable()` until it returns `sentinel` |

**Notes:** an iterator is **one-shot**: once consumed, looping again yields nothing. This bites when a generator is
passed to two functions, or when you call `len()`-free code twice over a `map` object. Materialise with `list()` if
you need several passes.

In [ ]:
import io

nums = [1, 2, 3]
it = iter(nums)
print(next(it), next(it), next(it), next(it, "done"))

squares = map(lambda x: x * x, nums)
print(sum(squares), sum(squares), "<- second pass is empty")

stream = io.StringIO("abcdefgh")
print(list(iter(lambda: stream.read(3), "")))

### Built-in iteration helpers

| function | returns | notes |
| --- | --- | --- |
| `enumerate(it, start=0)` | `(index, item)` pairs | |
| `zip(*its, strict=False)` | tuples, stopping at the shortest | `strict=True` (3.10+) raises `ValueError` on unequal lengths |
| `map(func, *its)` / `filter(func, it)` | lazy iterators | `filter(None, it)` keeps truthy items |
| `any(it)` / `all(it)` | bool, short-circuiting | `all([])` is True, `any([])` is False |
| `sum(it, start=0)`, `min/max(it, key=, default=)` | aggregates | `default=` avoids `ValueError` on empty input |
| `sorted(it, key=, reverse=)` / `reversed(seq)` | a new list / a reverse iterator | `reversed` needs a sequence |

**Notes:** `zip(*rows)` transposes a list of rows into columns. Prefer `strict=True` whenever unequal lengths would be
a bug: silent truncation is a common source of lost data.

In [ ]:
names, amounts = ["a", "b", "c"], [10, 20]
print(list(enumerate(names, start=1)), list(zip(names, amounts)))
try:
    list(zip(names, amounts, strict=True))
except ValueError as e:
    print("ValueError:", e)
rows = [(1, "a"), (2, "b"), (3, "c")]
print(list(zip(*rows)), list(filter(None, [0, 1, "", "x", None])))
print(all([]), any([]), max([], default=None), min(["bb", "a", "ccc"], key=len))

## Comprehensions

### List, dict, set and generator comprehensions

| form | builds |
| --- | --- |
| `[expr for x in it if cond]` | a list |
| `{k: v for k, v in pairs if cond}` | a dict (later keys win) |
| `{expr for x in it}` | a set |
| `(expr for x in it)` | a lazy generator; parentheses can be dropped as the only argument: `sum(x for x in it)` |
| `[a if cond else b for x in it]` | a conditional **expression** (a different place from the filter `if`) |
| `[f(x, y) for x in xs for y in ys]` | nested loops, outer first, as in a written-out `for` |

**Notes:** a comprehension has its own scope; its loop variable does not leak. Prefer a plain loop when the
comprehension needs more than one condition or nesting level to read.

In [ ]:
orders = [("north", 120), ("south", 0), ("north", 50), ("east", 70)]
print([amt for _, amt in orders if amt > 0])
print({region: amt for region, amt in orders})
print({region for region, _ in orders})
print(sum(amt for _, amt in orders), ["big" if amt >= 100 else "small" for _, amt in orders])
matrix = [[1, 2, 3], [4, 5, 6]]
print([x for row in matrix for x in row], [[row[i] for row in matrix] for i in range(3)])

## Generators

### Generator functions and `yield`

A function containing `yield` returns a **generator** when called; its body runs lazily, pausing at each `yield` and
resuming on the next `next()`. A `return value` ends it (the value becomes `StopIteration.value`). Generators hold
one item at a time, so they process files and streams larger than memory.

`yield from sub` delegates to another iterable, yielding all of its items (and returning its return value).

**Notes:** code before the first `yield` does not run until the first `next()`, which delays errors such as a missing
file. Use `try/finally` (or a `with` block) inside a generator to release resources if the consumer stops early;
`close()` raises `GeneratorExit` at the paused `yield`.

In [ ]:
def read_batches(rows, size):
    batch = []
    for row in rows:
        batch.append(row)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

print(list(read_batches(range(7), 3)), list(flatten([1, [2, [3, 4]], 5])))

def noisy():
    print("  started")
    try:
        yield 1
        yield 2
    finally:
        print("  cleaned up")

g = noisy()
print("created, nothing ran yet")
print(next(g))
g.close()

### Lazy pipelines

Chaining generators builds a streaming pipeline: each stage pulls one item at a time from the previous one, so memory
stays flat however large the input. This is the plain-Python version of a Spark job's narrow transformations.

In [ ]:
import sys

lines = (f"{i},{'paid' if i % 3 else 'void'},{i * 10}" for i in range(1_000_000))
parsed = (line.split(",") for line in lines)
paid = (int(amount) for _, status, amount in parsed if status == "paid")
print(sum(paid), "| generator object size:", sys.getsizeof(paid), "bytes vs list of 1e6 ints:", sys.getsizeof(list(range(1_000_000))))

## itertools

### Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`

| function | yields |
| --- | --- |
| `islice(it, stop)` / `islice(it, start, stop, step)` | a slice of any iterator, lazily (no negative indices) |
| `chain(a, b, ...)` / `chain.from_iterable(its)` | items of each iterable in turn |
| `takewhile(pred, it)` / `dropwhile(pred, it)` | items while `pred` holds / items after it first fails |
| `compress(data, selectors)` | items whose selector is truthy |
| `filterfalse(pred, it)` | items where `pred` is false |
| `count(start, step)`, `cycle(it)`, `repeat(x, n)` | infinite (or repeated) streams; bound them with `islice` or `zip` |

In [ ]:
from itertools import chain, compress, count, cycle, dropwhile, filterfalse, islice, repeat, takewhile

print(list(islice(count(100, 5), 3)), list(islice("abcdefg", 1, None, 2)))
print(list(chain([1, 2], (3,), "ab")), list(chain.from_iterable([[1], [2, 3]])))
print(list(takewhile(lambda x: x < 3, [1, 2, 5, 1])), list(dropwhile(lambda x: x < 3, [1, 2, 5, 1])))
print(list(compress("abcd", [1, 0, 1, 0])), list(filterfalse(str.isdigit, "a1b2")), list(zip(cycle("xy"), range(3))), list(repeat(0, 3)))

### Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`

| function | yields | notes |
| --- | --- | --- |
| `groupby(it, key=None)` | `(key, group_iterator)` for each **run** of equal keys | sort by the same key first to group globally; consume each group before advancing |
| `pairwise(it)` (3.10+) | overlapping pairs `(a, b), (b, c), ...` | differences, transitions |
| `batched(it, n)` (3.12+) | tuples of up to `n` items | chunking API calls and inserts; 3.13 adds `strict=` |
| `accumulate(it, func=operator.add, initial=None)` | running totals (or running max, min, product) | |

In [ ]:
import operator
from itertools import accumulate, batched, groupby, pairwise

events = [("north", 1), ("north", 2), ("south", 3), ("north", 4)]
print([(k, [v for _, v in g]) for k, g in groupby(events, key=lambda e: e[0])], "<- runs, not global groups")
print([(k, len(list(g))) for k, g in groupby(sorted(events), key=lambda e: e[0])])
print([b - a for a, b in pairwise([10, 13, 13, 20])], list(batched(range(7), 3)))
print(list(accumulate([3, 1, 4, 1, 5])), list(accumulate([3, 1, 4, 1, 5], max)), list(accumulate([1, 2, 3], operator.mul, initial=10)))

### Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`

| function | yields |
| --- | --- |
| `product(*its, repeat=1)` | the Cartesian product (nested loops) |
| `permutations(it, r=None)` | ordered arrangements of length `r` |
| `combinations(it, r)` / `combinations_with_replacement(it, r)` | unordered selections |
| `zip_longest(*its, fillvalue=None)` | like `zip`, padding the shorter inputs |
| `starmap(func, it)` | `func(*args)` for each tuple of args |
| `tee(it, n=2)` | `n` independent iterators over one source (buffers what one has read ahead) |

In [ ]:
from itertools import combinations, permutations, product, starmap, tee, zip_longest

print(list(product(["dev", "prod"], ["us", "eu"])), len(list(product(range(10), repeat=3))))
print(list(permutations("abc", 2))[:4], list(combinations("abcd", 2)))
print(list(zip_longest([1, 2, 3], ["a"], fillvalue="-")), list(starmap(pow, [(2, 3), (10, 2)])))
a, b2 = tee(iter([1, 2, 3]))
print(list(a), list(b2))

**Next:** practise these entries in [Python practice 6: iterators, generators and itertools](../practice/python-practice-06-iterators-and-itertools.ipynb).